In [1]:
import os
import yaml
import time
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from typing import List, Dict, Tuple
from huggingface_hub import login
from peft import PeftModel
from kaggle_secrets import UserSecretsClient

In [2]:
# ==================== КОНФИГУРАЦИЯ ====================
MODEL_1_ID = "sberbank-ai/rugpt3medium_based_on_gpt2"
MODEL_2_ID = "nikrog/rugpt3medium_finetuned_psychology"  # private
DATASET_ID = "nikrog/psychology_dataset_rus"  # замените на ваш датасет
MAX_TRAIN_SAMPLES = 150000
MAX_EVAL_SAMPLES = 500
MAX_SEQ_LENGTH = 512
PROMPT_TEMPLATE = "Ситуация: {situation}\nРекомендация психолога:"
TEST_CASES_PATH = "/kaggle/input/datasets/nikolayart/test-cases/test_cases.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
user_secrets = UserSecretsClient()
HGF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [3]:
# ==================== ЗАГРУЗКА ТЕСТОВЫХ СЛУЧАЕВ ====================
def load_test_cases(yaml_path: str) -> List[Dict]:
    """Загрузка тестовых случаев из YAML"""
    if not os.path.exists(yaml_path):
        raise FileNotFoundError(f"Файл {yaml_path} не найден.")
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
    return data.get('cases', [])

def load_and_tokenize_eval_data(tokenizer, dataset_id: str, max_samples: int = 500):
    """Загрузка и токенизация валидационной выборки для расчёта перплексии"""
    print("Загрузка датасета для оценки перплексии...")

    data_files = {
        "train": "train.txt",
        "test": "test.txt",
        "val": "val.txt"
    }
    
    # Адаптируйте split/data_files под вашу структуру датасета
    dataset = load_dataset(dataset_id, data_files=data_files)
    dataset = dataset["val"].select(range(min(len(dataset["val"]), max_samples)))
    
    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_SEQ_LENGTH
        )
    
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
    return tokenized  # set_format больше не нужен, конвертируем явно в цикле

def calculate_perplexity(model, tokenizer, dataset, batch_size: int = 4) -> float:
    """Расчёт средней перплексии модели на датасете"""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    
    with torch.inference_mode():
        for i in tqdm(range(0, len(dataset), batch_size), desc="Calculating perplexity"):
            batch = dataset[i:i+batch_size]
            
            # 🔑 Явно конвертируем списки в тензоры и переносим на устройство
            input_ids = torch.tensor(batch['input_ids'], device=DEVICE)
            attention_mask = torch.tensor(batch['attention_mask'], device=DEVICE)
            
            # Игнорируем паддинг-токены при расчёте loss
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            n_batches += 1
            
    avg_loss = total_loss / n_batches
    return np.exp(avg_loss)

# ==================== ГЕНЕРАЦИЯ ОТВЕТОВ С ЗАМЕРОМ ВРЕМЕНИ ====================
def generate_case_responses_with_timing(model, tokenizer, cases: List[Dict], 
                                        max_new_tokens: int = 256) -> List[Dict]:
    """Генерация ответов с измерением времени"""
    model.eval()
    responses = []
    print(f"Генерация ответов на {len(cases)} тестовых случаев...")
    
    for case in tqdm(cases, desc="Generating responses"):
        situation = case["situation"].strip()
        prompt = PROMPT_TEMPLATE.format(situation=situation)
        
        inputs = tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH
        ).to(DEVICE)
        
        # Замер времени генерации
        start_time = time.time()
        with torch.inference_mode():
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        elapsed_time = time.time() - start_time
        
        full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated = full_text[len(tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)):].strip()
        
        responses.append({
            "case_id": case["id"],
            "case_title": case["title"],
            "situation": situation,
            "generated_answer": generated,
            "reference_answer": case.get("reference_answer", ""),
            "evaluation_criteria": case.get("evaluation_criteria", {}),
            "generation_time_sec": round(elapsed_time, 3),
            "tokens_generated": len(outputs[0]) - inputs.input_ids.shape[1]
        })
    return responses

# ==================== ЗАГРУЗКА МОДЕЛИ ====================
def load_model_and_tokenizer(model_id: str, use_auth_token: bool = False):
    """Загрузка модели и токенизатора с обработкой приватных репозиториев"""
    print(f"Загрузка модели {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Добавляем pad_token если отсутствует
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    if model_id != 'sberbank-ai/rugpt3medium_based_on_gpt2':
        base_model = AutoModelForCausalLM.from_pretrained(
            "sberbank-ai/rugpt3medium_based_on_gpt2",
            device_map="auto",
            torch_dtype=torch.float16
        )
        model = PeftModel.from_pretrained(base_model, "nikrog/rugpt3medium_finetuned_psychology")
        model = model.merge_and_unload()

    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            low_cpu_mem_usage=True
        ).to(DEVICE)
    
    return model, tokenizer

# ==================== СРАВНЕНИЕ МОДЕЛЕЙ ====================
def compare_models():
    """Основная функция сравнения двух моделей"""
    
    # Загрузка тестовых случаев
    test_cases = load_test_cases(TEST_CASES_PATH)
    if not test_cases:
        raise ValueError("Тестовые случаи не найдены в файле")
    
    results = {}
    
    for model_idx, (model_id, is_private) in enumerate([
        (MODEL_1_ID, False), 
        (MODEL_2_ID, True)
    ], 1):
        print(f"\n{'='*60}")
        print(f"Оценка модели {model_idx}: {model_id}")
        print(f"{'='*60}")
        
        # Загрузка модели
        model, tokenizer = load_model_and_tokenizer(model_id, use_auth_token=is_private)
        
        # Расчёт перплексии
        print("\n[1/3] Расчёт перплексии...")
        eval_dataset = load_and_tokenize_eval_data(tokenizer, DATASET_ID, MAX_EVAL_SAMPLES)
        perplexity = calculate_perplexity(model, tokenizer, eval_dataset)
        print(f"Перплексия: {perplexity:.4f}")
        
        # Генерация ответов + замер времени
        print("\n[2/3] Генерация ответов на тестовые случаи...")
        responses = generate_case_responses_with_timing(model, tokenizer, test_cases)
        
        # Статистика по времени и токенам
        times = [r["generation_time_sec"] for r in responses]
        tokens = [r["tokens_generated"] for r in responses]
        avg_time = np.mean(times)
        avg_tokens_per_sec = np.mean(tokens) / avg_time if avg_time > 0 else 0
        
        print(f"Среднее время ответа: {avg_time:.3f} сек")
        print(f"Скорость генерации: {avg_tokens_per_sec:.2f} токенов/сек")
        
        results[model_id] = {
            "perplexity": perplexity,
            "avg_generation_time": avg_time,
            "tokens_per_second": avg_tokens_per_sec,
            "responses": responses
        }
        
        # Освобождение памяти
        del model, tokenizer
        torch.cuda.empty_cache()
    
    return results

# ==================== ВЫВОД РЕЗУЛЬТАТОВ ====================
def print_comparison_table(results: Dict):
    """Вывод сводной таблицы сравнения"""
    print(f"\n{'СВОДНАЯ ТАБЛИЦА СРАВНЕНИЯ':^60}")
    print(f"{'='*60}")
    
    data = []
    for model_id, metrics in results.items():
        data.append({
            "Модель": model_id.split('/')[-1],
            "Перплексия ↓": f"{metrics['perplexity']:.4f}",
            "Ср. время (сек) ↓": f"{metrics['avg_generation_time']:.3f}",
            "Токены/сек ↑": f"{metrics['tokens_per_second']:.2f}"
        })
    
    df = pd.DataFrame(data)
    print(df.to_string(index=False))
    
    # Рекомендация
    print(f"\n{'Рекомендация':^60}")
    models = list(results.keys())
    if results[models[0]]["perplexity"] < results[models[1]]["perplexity"]:
        print(f"• По перплексии лучше: {models[0].split('/')[-1]}")
    else:
        print(f"• По перплексии лучше: {models[1].split('/')[-1]}")
    
    if results[models[0]]["avg_generation_time"] < results[models[1]]["avg_generation_time"]:
        print(f"• По скорости лучше: {models[0].split('/')[-1]}")
    else:
        print(f"• По скорости лучше: {models[1].split('/')[-1]}")

def save_responses(results: Dict, output_dir: str = "comparison_results"):
    """Сохранение сгенерированных ответов для ручного анализа"""
    os.makedirs(output_dir, exist_ok=True)
    
    for model_id, data in results.items():
        model_name = model_id.replace('/', '_')
        path = os.path.join(output_dir, f"{model_name}_responses.yaml")
        
        with open(path, 'w', encoding='utf-8') as f:
            # Убран ensure_ascii=False (это параметр для json)
            # default_flow_style=False делает вывод более читаемым
            yaml.dump({
                "model": model_id,
                "perplexity": data["perplexity"],
                "avg_generation_time": data["avg_generation_time"],
                "responses": data["responses"]
            }, f, default_flow_style=False, allow_unicode=True)
            
        print(f"Ответы сохранены: {path}")

In [4]:
login(token=HGF_TOKEN)
print("Success authentification in Huggingface Hub")

results = compare_models()
print_comparison_table(results)
save_responses(results)

# Опционально: сохранить таблицу в CSV
pd.DataFrame([
    {
        "model": mid,
        "perplexity": res["perplexity"],
        "avg_time_sec": res["avg_generation_time"],
        "tokens_per_sec": res["tokens_per_second"]
    }
    for mid, res in results.items()
]).to_csv("comparison_summary.csv", index=False, sep=';')
print("Сводка сохранена в comparison_summary.csv")

Success authentification in Huggingface Hub

Оценка модели 1: sberbank-ai/rugpt3medium_based_on_gpt2
Загрузка модели sberbank-ai/rugpt3medium_based_on_gpt2...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Calculating perplexity: 100%|██████████| 125/125 [00:14<00:00,  8.84it/s]


Перплексия: 3197.7933

[2/3] Генерация ответов на тестовые случаи...
Генерация ответов на 10 тестовых случаев...


Generating responses: 100%|██████████| 10/10 [00:59<00:00,  5.95s/it]

Среднее время ответа: 5.944 сек
Скорость генерации: 43.07 токенов/сек

                 СВОДНАЯ ТАБЛИЦА СРАВНЕНИЯ                  
                           Модель Перплексия ↓ Ср. время (сек) ↓ Токены/сек ↑
       rugpt3medium_based_on_gpt2  252446.5800             4.478        57.16
rugpt3medium_finetuned_psychology    3197.7933             5.944        43.07

                        Рекомендация                        
• По перплексии лучше: rugpt3medium_finetuned_psychology
• По скорости лучше: rugpt3medium_based_on_gpt2
Ответы сохранены: comparison_results/sberbank-ai_rugpt3medium_based_on_gpt2_responses.yaml
Ответы сохранены: comparison_results/nikrog_rugpt3medium_finetuned_psychology_responses.yaml
Сводка сохранена в comparison_summary.csv
